In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("netflix_titles.csv")

print(df.head())
print(df.tail())

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.info()

print(df.describe().T)

release_year = df["release_year"]

print("Earliest year:", release_year.min())
print("Latest year:", release_year.max())
print("Average year:", release_year.mean())

In [ ]:
selected = df[
    ["title", "type", "country", "release_year", "rating"]
]

recent_movies = df[
    (df["type"] == "Movie") &
    (df["release_year"] > 2015)
]

old_shows = df[
    (df["type"] == "TV Show") &
    (df["release_year"] < 2010)
]

love_titles = df[
    df["title"].str.contains("love", case=False, na=False)
]

both_types = df[df["type"].isin(["Movie", "TV Show"])]

print(recent_movies.head())
print(old_shows.head())
print(love_titles[["title", "type"]].head())
print(both_types.shape)

In [ ]:
print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())

cleaned = df.copy()

cleaned["country"] = cleaned["country"].fillna("Unknown")
cleaned["rating"] = cleaned["rating"].fillna(
    cleaned["rating"].mode()[0]
)

cleaned = cleaned.drop(columns=["director"])

print(cleaned.isnull().sum())

In [ ]:
print(cleaned["type"].unique())
print(cleaned["type"].value_counts())
print(cleaned["release_year"].value_counts().head(10))
titles_per_year = cleaned.groupby("release_year").size()
print(titles_per_year)
cleaned["title_length"] = cleaned["title"].str.len()
print(cleaned[["title", "title_length"]].head())

year_type = pd.pivot_table(
    cleaned,
    index="release_year",
    columns="type",
    values="show_id",
    aggfunc="count",
    fill_value=0
)

print(year_type.head())

In [ ]:
type_labels = pd.DataFrame({
    "type": ["Movie", "TV Show"],
    "category": ["Film", "Series"]
})

merged = pd.merge(
    cleaned,
    type_labels,
    on="type",
    how="left"
)

print(merged[["title", "type", "category"]].head())

cleaned["type"].value_counts().plot(kind="bar")
plt.title("Movies vs TV Shows")
plt.xlabel("Content Type")
plt.ylabel("Number of Titles")
plt.xticks(rotation=0)
plt.show()


plt.hist(cleaned["release_year"].dropna(), bins=20)
plt.title("Distribution of Release Years")
plt.xlabel("Release Year")
plt.ylabel("Number of Titles")
plt.show()

plot_data = cleaned.dropna(
    subset=["title_length", "release_year"]
)

plt.scatter(
    plot_data["release_year"],
    plot_data["title_length"],
    alpha=0.3
)

plt.title("Title Length vs Release Year")
plt.xlabel("Release Year")
plt.ylabel("Title Length (characters)")
plt.show()

In [ ]:
cleaned.to_csv("netflix_cleaned.csv", index=False)
print("Cleaned dataset saved!")